In [1]:
import os

os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"
from transformers import LlamaTokenizer
from sentencepiece import sentencepiece_model_pb2 as sp_pb2_model
import sentencepiece as spm

In [3]:
llama_tokenizer_dir = "meta-llama/Llama-2-7b-chat-hf"
chinese_sp_model_file = "../checkpoints/malayalam-10k-mini/malayalam-10k.model"
llama_tokenizer = LlamaTokenizer.from_pretrained(llama_tokenizer_dir)
chinese_sp_model = spm.SentencePieceProcessor()
chinese_sp_model.Load(chinese_sp_model_file)

True

In [4]:
llama_spm = sp_pb2_model.ModelProto()
llama_spm.ParseFromString(llama_tokenizer.sp_model.serialized_model_proto())
chinese_spm = sp_pb2_model.ModelProto()
chinese_spm.ParseFromString(chinese_sp_model.serialized_model_proto())

428768

In [5]:
print(len(llama_tokenizer), len(chinese_sp_model))
print(llama_tokenizer.all_special_tokens)
print(llama_tokenizer.all_special_ids)
print(llama_tokenizer.special_tokens_map)

32000 10000
['<s>', '</s>', '<unk>']
[1, 2, 0]
{'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>'}


In [6]:
llama_spm_tokens_set = set(p.piece for p in llama_spm.pieces)
len(llama_spm_tokens_set)

32000

In [10]:
for p in chinese_spm.pieces:
    piece = p.piece
    if piece not in llama_spm_tokens_set:
        new_p = sp_pb2_model.ModelProto().SentencePiece()
        new_p.piece = piece
        new_p.score = 0
        llama_spm.pieces.append(new_p)
len(llama_spm.pieces)

39590

In [11]:
output_sp_dir = "merged_tokenizer_sp"
output_hf_dir = "merged_tokenizer_hf"  # the path to save Chinese-LLaMA tokenizer
os.makedirs(output_sp_dir, exist_ok=True)

In [12]:
!ls

merge.ipynb         merged_tokenizer_sp tokenizer.ipynb


In [13]:
with open(output_sp_dir + "/chinese_llama.model", "wb") as f:
    f.write(llama_spm.SerializeToString())
tokenizer = LlamaTokenizer(vocab_file=output_sp_dir + "/chinese_llama.model")

You are using the default legacy behaviour of the <class 'transformers.models.llama.tokenization_llama.LlamaTokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


In [14]:
!tree

.
├── merge.ipynb
├── merged_tokenizer_sp
│   └── chinese_llama.model
└── tokenizer.ipynb

2 directories, 3 files


In [15]:
tokenizer.save_pretrained(output_hf_dir)
print(f"Chinese-LLaMA tokenizer has been saved to {output_hf_dir}")

Chinese-LLaMA tokenizer has been saved to merged_tokenizer_hf


In [17]:
!tree

.
├── merge.ipynb
├── merged_tokenizer_hf
│   ├── special_tokens_map.json
│   ├── tokenizer.model
│   └── tokenizer_config.json
├── merged_tokenizer_sp
│   └── chinese_llama.model
└── tokenizer.ipynb

3 directories, 6 files


In [24]:
llama_tokenizer = LlamaTokenizer.from_pretrained(llama_tokenizer_dir)
chinese_llama_tokenizer = LlamaTokenizer.from_pretrained(output_hf_dir)
print(tokenizer.all_special_tokens)
print(tokenizer.all_special_ids)
print(tokenizer.special_tokens_map)
text = """ജിപ്സീസ് എന്ന ഒരിടത്തും നിൽക്കാത്ത ഒരു ജനത അന്നു അക്കാലത്തും ആ റഷ്യയിൽ ഉണ്ടായിരുന്നു എന്നത് കാരമസോവ് ബ്രദേഴ്സ് വായിച്ചവർക്ക് അറിയാം. വേശ്യകളെ പുനരധിവസിപ്പിക്കാൻ ധാരാളം ശ്രമങ്ങൾ നടന്നിരുന്നു. മാത്രമല്ല വിശുദ്ധ വേശ്യകളെ ചിത്രീകരിച്ചുകൊണ്ട് അക്കാലത്തെ ധാരാളം സാഹിത്യകൃതികൾ വന്നിരുന്നു. 1863ൽ നിക്കോളായി ചെർണിഷേവ്സ്കിയുടെ വാട്ട് ഈ റ്റു ബി ഡൺ? എന്ന നോവൽ ഉദാഹരണം. ദൊസ്റ്റോയോവ്സ്കി വിക്റ്റർ ഹ്യൂഗോയുടെ ഒരു ആരാധകൻ ആയിരുന്നു എന്ന് ജോസഫ് ഫ്രാങ്ക്, ദൊസ്റ്റോയോവ്സ്കിയുടെ ജീവചരിത്രകാരൻ പറയുന്നു. ഹ്യൂഗോയുടെ സോഷ്യൽ ഹ്യുമാനിറ്റേറിയനിസം ദൊസ്റ്റോയോവ്സ്കിയെ സ്വാധീനിച്ചിരുന്നു. പുഷ്കിൻ ആയിരുന്നു മറ്റൊരു ഫേവറൈറ്റ് റൈറ്റർ.
The primary use of LLaMA is research on large language models, including"""
print("Test text:\n", text)
print(f"Tokenized by LLaMA tokenizer:\n{llama_tokenizer.tokenize(text)}")
print(f"+++++")
print(
    f"Tokenized by Chinese-LLaMA tokenizer:\n{chinese_llama_tokenizer.tokenize(text)}"
)

['<s>', '</s>', '<unk>']
[1, 2, 0]
{'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>'}
Test text:
 ജിപ്സീസ് എന്ന ഒരിടത്തും നിൽക്കാത്ത ഒരു ജനത അന്നു അക്കാലത്തും ആ റഷ്യയിൽ ഉണ്ടായിരുന്നു എന്നത് കാരമസോവ് ബ്രദേഴ്സ് വായിച്ചവർക്ക് അറിയാം. വേശ്യകളെ പുനരധിവസിപ്പിക്കാൻ ധാരാളം ശ്രമങ്ങൾ നടന്നിരുന്നു. മാത്രമല്ല വിശുദ്ധ വേശ്യകളെ ചിത്രീകരിച്ചുകൊണ്ട് അക്കാലത്തെ ധാരാളം സാഹിത്യകൃതികൾ വന്നിരുന്നു. 1863ൽ നിക്കോളായി ചെർണിഷേവ്സ്കിയുടെ വാട്ട് ഈ റ്റു ബി ഡൺ? എന്ന നോവൽ ഉദാഹരണം. ദൊസ്റ്റോയോവ്സ്കി വിക്റ്റർ ഹ്യൂഗോയുടെ ഒരു ആരാധകൻ ആയിരുന്നു എന്ന് ജോസഫ് ഫ്രാങ്ക്, ദൊസ്റ്റോയോവ്സ്കിയുടെ ജീവചരിത്രകാരൻ പറയുന്നു. ഹ്യൂഗോയുടെ സോഷ്യൽ ഹ്യുമാനിറ്റേറിയനിസം ദൊസ്റ്റോയോവ്സ്കിയെ സ്വാധീനിച്ചിരുന്നു. പുഷ്കിൻ ആയിരുന്നു മറ്റൊരു ഫേവറൈറ്റ് റൈറ്റർ.
The primary use of LLaMA is research on large language models, including
Tokenized by LLaMA tokenizer:
['▁', '<0xE0>', '<0xB4>', '<0x9C>', 'ി', 'പ', '്', 'സ', '<0xE0>', '<0xB5>', '<0x80>', 'സ', '്', '▁', 'എ', 'ന', '്', 'ന', '▁', '<0xE0>', '<0xB4>', '<0x92>', 'ര', 'ി', 'ട', 'ത', '്', '